# Card Data Collector

Builds a YOLO dataset of our own cards from the robot's camera poses.

How it works:
1. The arm goes to a pose (`mycards` or `dealer`) from `poses.json`.
2. It takes a picture of the **empty table** (the background).
3. You place **one card**, type its rank, hit Enter.
4. It finds the card by comparing to the empty picture, draws the box, and saves the image + YOLO label.

Ranks: `A 2 3 4 5 6 7 8 9 10 J Q K`, plus `DOWN` for a face-down card (like the dealer's hole card). Suit doesn't matter for blackjack.

Output goes to `card_dataset/` (images, labels, `data.yaml`), ready to train on a laptop/Colab.

In [ ]:
import json
import os
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from Arm_Lib import Arm_Device

arm = Arm_Device()

CAMERA_INDEX = 0   # try 1 if you get no frames
cap = cv2.VideoCapture(CAMERA_INDEX)
time.sleep(1)      # let the camera warm up

with open('poses.json') as f:
    poses = json.load(f)
print("Poses:", list(poses))

## Dataset setup
Class order matters: this is the order YOLO will use.

In [ ]:
CLASSES = ['A', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'DOWN']   # DOWN = face-down card

DATA_DIR = 'card_dataset'
os.makedirs(f'{DATA_DIR}/images', exist_ok=True)
os.makedirs(f'{DATA_DIR}/labels', exist_ok=True)

with open(f'{DATA_DIR}/data.yaml', 'w') as f:
    f.write(f"path: {DATA_DIR}\ntrain: images\nval: images\n")
    f.write(f"nc: {len(CLASSES)}\n")
    f.write("names: [" + ", ".join(f"'{c}'" for c in CLASSES) + "]\n")

print("Images so far:", len(os.listdir(f'{DATA_DIR}/images')))

## Helper functions

In [ ]:
def move(q):
    # 6th value is the gripper (held at 90), 1500 = take 1.5 seconds
    arm.Arm_serial_servo_write6(*q, 90, 1500)
    time.sleep(2)


def take_picture():
    global cap
    if not cap.isOpened():
        cap = cv2.VideoCapture(CAMERA_INDEX)
        time.sleep(1)
    # Throw away a few old frames so we get the current view
    for _ in range(5):
        cap.read()
    ok, frame = cap.read()
    return frame if ok else None


def find_card(background, frame, min_area=1500):
    """Box around the biggest thing that changed since the empty picture."""
    diff = cv2.absdiff(cv2.GaussianBlur(background, (5, 5), 0),
                       cv2.GaussianBlur(frame, (5, 5), 0))
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    contours = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    if not contours:
        return None
    biggest = max(contours, key=cv2.contourArea)
    if cv2.contourArea(biggest) < min_area:
        return None
    return cv2.boundingRect(biggest)   # x, y, w, h in pixels


def show(frame, box=None, title=''):
    img = frame.copy()
    if box is not None:
        x, y, w, h = box
        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
    plt.figure(figsize=(6, 4.5))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()


def save_sample(frame, box, rank, pose_name):
    h_img, w_img = frame.shape[:2]
    x, y, w, h = box
    # YOLO format: class x_center y_center width height (all 0-1)
    line = f"{CLASSES.index(rank)} {(x + w/2)/w_img:.6f} {(y + h/2)/h_img:.6f} {w/w_img:.6f} {h/h_img:.6f}\n"
    name = f"{pose_name}_{rank}_{int(time.time()*1000)}"
    cv2.imwrite(f"{DATA_DIR}/images/{name}.jpg", frame)
    with open(f"{DATA_DIR}/labels/{name}.txt", 'w') as f:
        f.write(line)
    return name

## Collect data
Set `POSE` to `'mycards'` or `'dealer'` and run. Then at the prompt:
- type a rank (`A`, `2`...`10`, `J`, `Q`, `K`) and Enter = save the card you placed
- `b` = retake the empty-table picture (do this if lighting changes or the table moves)
- `u` = undo the last save
- just Enter (empty) = quit  (`Q` means queen)
- `1` or `T` also count as `10`
- `D` = face-down card (`DOWN`)

**Tips for a good dataset:** take each card a few times in slightly different spots and angles. Do the same cards in both poses. Aim for ~10 shots per rank per pose.

In [ ]:
POSE = 'mycards'

move(poses[POSE])
input("Clear the table, then hit Enter to take the empty picture...")
background = take_picture()
show(background, title='Empty table')

last = None
counts = {}
while True:
    key = input("Place ONE card, type its rank (A 2-10 J Q K, D = face down) | b = new background | u = undo | just Enter = quit: ").strip().upper()
    if key == '':
        break
    if key == 'B':
        input("Clear the table, then hit Enter...")
        background = take_picture()
        show(background, title='Empty table')
        continue
    if key == 'U':
        if last:
            os.remove(f"{DATA_DIR}/images/{last}.jpg")
            os.remove(f"{DATA_DIR}/labels/{last}.txt")
            print("Removed", last)
            last = None
        continue
    if key == 'D':
        key = 'DOWN'
    if key == '1' or key == 'T':
        key = '10'
    if key not in CLASSES:
        print("Not a rank:", key)
        continue

    frame = take_picture()
    if frame is None:
        print("No camera frame, try again")
        continue
    box = find_card(background, frame)
    if box is None:
        print("Couldn't find a card. Is it in view? Try 'b' for a new background.")
        show(frame, title='No card found')
        continue

    last = save_sample(frame, box, key, POSE)
    counts[key] = counts.get(key, 0) + 1
    show(frame, box, f"{key}  (saved {counts[key]} this session)")

print("This session:", counts)
print("Total images:", len(os.listdir(f'{DATA_DIR}/images')))

## Check the dataset
Shows a few random saved images with their boxes so you can catch bad labels.

In [ ]:
import random

names = [n[:-4] for n in os.listdir(f'{DATA_DIR}/images') if n.endswith('.jpg')]
for name in random.sample(names, min(6, len(names))):
    frame = cv2.imread(f"{DATA_DIR}/images/{name}.jpg")
    h_img, w_img = frame.shape[:2]
    c, xc, yc, w, h = open(f"{DATA_DIR}/labels/{name}.txt").read().split()
    xc, yc, w, h = float(xc)*w_img, float(yc)*h_img, float(w)*w_img, float(h)*h_img
    show(frame, (int(xc - w/2), int(yc - h/2), int(w), int(h)), f"{CLASSES[int(c)]}  {name}")

counts = {}
for n in os.listdir(f'{DATA_DIR}/labels'):
    c = CLASSES[int(open(f'{DATA_DIR}/labels/{n}').read().split()[0])]
    counts[c] = counts.get(c, 0) + 1
print("Images per rank:", {c: counts.get(c, 0) for c in CLASSES})

## Done

In [ ]:
move(poses['home'])
cap.release()